# HomeC Anomaly Detection — Corrected Model Pipeline

This notebook documents the corrected modeling workflow. The executable source of truth is `../src/train_model.py` so the notebook, saved artifacts, dashboard, and reported metrics cannot drift apart.

## Corrections in this revision

- Uses a **chronological 70% / 15% / 15% split** instead of random shuffling.
- Does **not** apply SMOTE to time-series observations.
- Removes the label source `use [kW]` from the feature matrix.
- Uses exactly eight deployable features that are also available in the dashboard.
- Selects the model and decision threshold on validation data only.
- Re-fits the selected configuration and evaluates the held-out test period once.
- Saves the exact evaluated model, feature order, defaults, metadata, and comparison table.

> The anomaly target remains a statistical proxy: `use [kW] > mean + 3×std`. It is not a verified equipment-fault label.

In [1]:
from pathlib import Path
import json
import joblib
import pandas as pd

ROOT = Path('..').resolve()
DATA_PATH = ROOT / 'data' / 'HomeC_cleaned_final.zip'
MODEL_DIR = ROOT / 'notebooks'

df = pd.read_csv(DATA_PATH, low_memory=False)
df['datetime'] = pd.to_datetime(df['datetime'])
print('Shape:', df.shape)
print('Period:', df['datetime'].min(), '->', df['datetime'].max())

Shape: (503910, 40)
Period: 2016-01-01 00:00:00 -> 2016-12-15 22:29:00


## Deployable feature set

The model uses only fields that the dashboard can provide:

1. `gen_kw` (renamed from `gen [kW]` for XGBoost compatibility)
2. `total_appliance`
3. `temperature`
4. `humidity`
5. `hour`
6. `dayofweek`
7. `month`
8. `is_weekend`

The direct label source `use [kW]`, timestamps, raw text categories, and dozens of unfilled appliance columns are excluded.

In [2]:
SOURCE_FEATURES = [
    'gen [kW]', 'total_appliance', 'temperature', 'humidity',
    'hour', 'dayofweek', 'month', 'is_weekend'
]
threshold_kw = df['use [kW]'].mean() + 3 * df['use [kW]'].std()
y = (df['use [kW]'] > threshold_kw).astype('int8')
X = df[SOURCE_FEATURES].rename(columns={'gen [kW]': 'gen_kw'})

n = len(df)
train_end, val_end = int(n * 0.70), int(n * 0.85)
print('Threshold:', round(threshold_kw, 6), 'kW')
print('Train / validation / test:', train_end, val_end-train_end, n-val_end)
print('Overall anomaly rate:', round(y.mean()*100, 2), '%')

Threshold: 4.033584 kW
Train / validation / test: 352737 75586 75587
Overall anomaly rate: 2.46 %


## Train and export artifacts

Run the cell below from the `notebooks/` directory. It executes the audited training script and overwrites all model artifacts consistently.

The script compares Logistic Regression, Random Forest, XGBoost, LightGBM, and Isolation Forest. To keep retraining practical on a student machine, it uses evenly spaced rows across the complete training period rather than taking a random sample.

In [ ]:
# Re-run when you need to rebuild the model artifacts.
# %run ../src/train_model.py

## Validation comparison from the latest completed run

In [3]:
comparison = pd.read_csv(MODEL_DIR / 'model_comparison.csv')
comparison

Model,F1,AUC,Precision,Recall,DecisionThreshold
LightGBM,0.2987,0.9511,0.2241,0.4477,0.450795
RandomForest,0.2424,0.9287,0.1445,0.7503,0.139608
XGBoost,0.2222,0.9435,0.1427,0.5017,0.644966
LogisticRegression,0.2131,0.923,0.9304,0.1204,0.999999
IsolationForest,0.1381,0.7529,0.1453,0.1316,0.602938


## Final held-out test result

In [4]:
metadata = json.loads((MODEL_DIR / 'model_metadata.json').read_text())
metadata['selected_model'], metadata['decision_threshold'], metadata['test_metrics'], metadata['confusion_matrix']

('LightGBM', 0.45079543, {'F1': 0.2912, 'AUC': 0.8987, 'Precision': 0.2648, 'Recall': 0.3235}, [[74748, 397], [299, 143]])

### Interpretation

The selected model is **LightGBM**, chosen using validation F1 with a calibrated decision threshold of **0.4508**. On the untouched chronological test period, it achieved:

- F1: **0.2912**
- ROC-AUC: **0.8987**
- Precision: **0.2648**
- Recall: **0.3235**

This result is substantially lower than the old random-split score, but it is more credible because future observations are not mixed into training. The remaining performance gap reflects temporal distribution shift and the weak proxy nature of the label.

In [5]:
model = joblib.load(MODEL_DIR / 'best_model.pkl')
feature_columns = joblib.load(MODEL_DIR / 'feature_columns.pkl')
defaults = json.loads((MODEL_DIR / 'feature_defaults.json').read_text())

row = pd.DataFrame([defaults]).reindex(columns=feature_columns)
probability = float(model.predict_proba(row)[:, 1][0])
prediction = int(probability >= metadata['decision_threshold'])
print({'probability': round(probability, 4), 'prediction': prediction})

## Limitations

- The label is generated from a fixed statistical threshold, so it represents unusually high load rather than confirmed faults.
- `total_appliance` is closely related to household consumption and is expected to dominate prediction.
- The original time counter is unreliable; dates are reconstructed under the one-reading-per-minute assumption.
- Results are currently validated on one household only, so they should not be generalized to other homes without external validation.